# Spot-Forward Spread Analysis: OU-GARCH Model

Analysis of electricity spot-forward spread using Ornstein-Uhlenbeck process (mean reversion) and GARCH modeling (volatility dynamics).

In [0]:
import dataiku
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
from arch import arch_model
import warnings

warnings.filterwarnings('ignore')

## 1. Load Data

In [0]:
# Load datasets
df_fw = dataiku.Dataset("forward_curve").get_dataframe()
df_spot = dataiku.Dataset("VPP_NET_VAL_DETAILS_VW_copy").get_dataframe()

print(f"Forward data shape: {df_fw.shape}")
print(f"Spot data shape: {df_spot.shape}")

## 2. Prepare Data

In [0]:
# Spot: daily average price
df_spot['dhis'] = df_spot['start_ts_utc'].dt.strftime('%Y-%m-%d')
df_spot = df_spot.groupby(['dhis'], as_index=False)['da_price'].mean()

print("Spot data ready:")
print(df_spot.head())

In [0]:
# Forward: BM futures only
df_fw = (
    df_fw[df_fw['product'].str.contains('BM', case=False, na=False)]
    [['dhis', 'dbeg', 'price_settle']]
    .copy()
)

# Convert to datetime
df_fw['dhis'] = pd.to_datetime(df_fw['dhis'])
df_fw['dbeg'] = pd.to_datetime(df_fw['dbeg'])

print("\nForward data ready:")
print(df_fw.head())

## 3. Merge & Calculate Spread

Link each spot price to its corresponding forward contract (same month).
Calculate log-spread: ln(spot/forward)

In [0]:
# Prepare for merge
df_spot['dhis'] = pd.to_datetime(df_spot['dhis'])
df_spot['dmonth'] = df_spot['dhis'].dt.to_period('M').dt.to_timestamp()

# Merge spot and forward
df = df_spot.merge(
    df_fw,
    how='left',
    left_on=['dhis', 'dmonth'],
    right_on=['dhis', 'dbeg']
)

df = df.drop(columns=['dbeg'])

print(f"Merged data shape: {df.shape}")

In [0]:
# Calculate log-spread and filter
df['spread_spot_fw'] = np.log(df['da_price'] / df['price_settle'])
df['spread_spot_fw'].fillna(0, inplace =True)
df = df[df['dhis'] > '2023'].copy()

print(f"Observations after filtering: {len(df)}")
print(f"\nSpread statistics:")
print(df['spread_spot_fw'].describe())

In [0]:
plt.plot(df['dhis'], df['spread_spot_fw'])

## 4. Stationarity Test

**ADF Test**: H₀ = unit root (non-stationary) | p < 0.05 → stationary ✓

In [0]:
# ADF test
adf = adfuller(df['spread_spot_fw'].dropna())
p_value = adf[1]

print(f"ADF Test p-value: {p_value:.6f}")
if p_value < 0.05:
    print("✓ Series is stationary (mean-reverting)")
else:
    print("✗ Series is non-stationary")
    

In [0]:
# ACF plot
plt.figure(figsize=(12, 5))
plot_acf(df['spread_spot_fw'].dropna(), lags=60)
plt.title('Autocorrelation Function')
plt.tight_layout()
plt.show()

## 5. Ornstein-Uhlenbeck Model

**Discrete OU (AR(1))**: $$X_t = \alpha + \beta X_{t-1} + \epsilon_t$$

**Parameter Recovery**:
- $\hat{\kappa} = -\ln(\hat{\beta})/\Delta t$ (mean reversion speed, year⁻¹)
- $\hat{\mu} = \hat{\alpha}/(1 - \hat{\beta})$ (long-term equilibrium)
- Half-life = $\ln(2)/\kappa \times 365$ (days)

In [0]:
# AR(1) regression
x = df['spread_spot_fw'].fillna(0)
y = x.iloc[1:].values
X = sm.add_constant(x.iloc[:-1].values)

model = sm.OLS(y, X).fit()

alpha = model.params[0]
beta = model.params[1]
residuals = model.resid

print(model.summary())

In [0]:
# Recover OU parameters
dt = 1 / 365

kappa = -np.log(beta) / dt
mu = alpha / (1 - beta)
half_life_days = np.log(2) / kappa * 365

# Volatility
sigma_eps = residuals.std(ddof=1)
sigma = sigma_eps * np.sqrt(
    2 * kappa / (1 - np.exp(-2 * kappa * dt))
)

print("\n" + "="*50)
print("OU MODEL PARAMETERS")
print("="*50)
print(f"Mean reversion speed (κ): {kappa:.4f}")
print(f"Long-term mean (μ):       {mu:.6f}")
print(f"Half-life:                {half_life_days:.2f} days")
print(f"Volatility (σ):           {sigma:.6f}")
print("="*50)

## 6. Residual Analysis

In [0]:
# Distribution statistics
sk = skew(residuals)
kt = kurtosis(residuals, fisher=False)

print(f"Skewness:  {sk:.4f}")
print(f"Kurtosis:  {kt:.4f}")
print(f"\nResiduals:\n{ pd.DataFrame(residuals).describe()}")

In [0]:
# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(residuals, bins=50, stat='density', kde=True, ax=axes[0])
axes[0].set_title('Distribution of Residuals')

sm.qqplot(residuals, line='45', ax=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

## 7. GARCH Effects Test

**Ljung-Box on squared residuals**: p < 0.05 → GARCH effects present

In [0]:
# Ljung-Box test
lb = acorr_ljungbox(residuals**2, lags=[10, 20, 30], return_df=True)
print("Ljung-Box Test (squared residuals):")
print(lb)

## 8. GARCH(1,1) Model

**Conditional Variance**: $\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$

In [0]:
# GARCH Gaussian
garch = arch_model(
    residuals,
    mean='Zero',
    vol='GARCH',
    p=1,
    q=1,
    dist='Normal'
)

garch_fit = garch.fit(disp='off')
print("GARCH(1,1) - Gaussian:")
print(garch_fit.summary())

In [0]:
# GARCH Student-t
garch_t = arch_model(
    residuals,
    mean='Zero',
    vol='GARCH',
    p=1,
    q=1,
    dist='t'
)

garch_t_fit = garch_t.fit(disp='off')
print("\nGARCH(1,1) - Student-t:")
print(garch_t_fit.summary())

In [0]:
# Model comparison
print("\nModel Comparison:")
print(f"Gaussian  | AIC: {garch_fit.aic:.2f} | BIC: {garch_fit.bic:.2f}")
print(f"Student-t | AIC: {garch_t_fit.aic:.2f} | BIC: {garch_t_fit.bic:.2f}")

if garch_t_fit.aic < garch_fit.aic:
    print("→ Student-t preferred")
else:
    print("→ Gaussian preferred")

## 9. Monte-Carlo Simulation

In [0]:
def simulate_ou(x0, mu, kappa, sigma, horizon_days, n_sims):
    """Simulate OU process paths (Euler discretization)."""
    dt = 1 / 365
    paths = np.zeros((horizon_days + 1, n_sims))
    paths[0] = x0
    
    exp_term = np.exp(-kappa * dt)
    vol_term = sigma * np.sqrt(
        (1 - np.exp(-2 * kappa * dt)) / (2 * kappa)
    )
    
    for t in range(horizon_days):
        z = np.random.normal(size=n_sims)
        paths[t + 1] = mu + (paths[t] - mu) * exp_term + vol_term * z
    
    return paths

# Generate simulations
n_days = len(df)
paths = simulate_ou(
    x0=df['spread_spot_fw'].iloc[-1],
    mu=mu,
    kappa=kappa,
    sigma=sigma,
    horizon_days=n_days - 1,
    n_sims=100
)

print(f"Simulated {paths.shape[1]} paths over {paths.shape[0]-1} days")

In [0]:
# Convert spread to spot price
spot_paths = df['price_settle'].values[:, None] * np.exp(paths)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(spot_paths, color='blue', alpha=0.3, linewidth=0.8)
plt.xlabel('Time Step (days)')
plt.ylabel('Spot Price')
plt.title('Monte-Carlo Simulation: 100 Spot Price Paths')
plt.tight_layout()
plt.show()

## 10. Risk Analysis

In [0]:
# Quantile comparison
quantiles = [0.50, 0.90, 0.95, 0.99]
actual_q = df['da_price'].quantile(quantiles)
simulated_q = pd.Series(spot_paths.flatten()).quantile(quantiles)

comparison = pd.DataFrame({
    'Quantile': [f'{q:.0%}' for q in quantiles],
    'Actual': actual_q.values,
    'Simulated': simulated_q.values,
    'Difference': (simulated_q.values - actual_q.values)
})

print("Quantile Comparison:")
print(comparison.to_string(index=False))

In [0]:
# Rolling volatility
rolling_std = df['spread_spot_fw'].fillna(0).rolling(90).std()

plt.figure(figsize=(14, 5))
rolling_std.plot(linewidth=2)
plt.xlabel('Date')
plt.ylabel('Standard Deviation')
plt.title('90-Day Rolling Volatility')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()